# 🚀 SROIE GPU Pipeline on Google Colab
This notebook is configured to run your SROIE Agentic Pipeline using a free NVIDIA T4 GPU on Google Colab.

### **Step 1: Enable the GPU**
1. Go to **Runtime** > **Change runtime type** in the top menu.
2. Select **T4 GPU** under Hardware accelerator and click Save.
3. Run the cell below to verify the GPU is active.

In [ ]:
!nvidia-smi

### **Step 2: Upload Your Project Files**
Since your project is local, the easiest way to bring it here is to zip it up:
1. Zip your entire `sroie-pipeline` folder on your computer (call it `sroie-pipeline.zip`).
2. Click the **Folder icon** on the left sidebar of Colab.
3. Click the **Upload** icon and select your `sroie-pipeline.zip` file.
4. Run the cell below to unzip it.

In [ ]:
!unzip -q sroie-pipeline.zip -d /content/
%cd /content/sroie-pipeline
!ls

### **Step 3: Install Dependencies & GPU Libraries**
This installs your requirements and ensures PyTorch uses the GPU (CUDA).

In [ ]:
!pip install -r requirements.txt
# Install the CUDA version of PyTorch for Docling
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

### **Step 4: Configure GPU for Docling**
We need to update your `parsing_agent.py` to tell it to use the GPU. This script does it automatically for you.

In [ ]:
parsing_agent_code = """import os
from parser.document_parser import parse_sroie_box_file
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PipelineOptions, RapidOcrOptions
from docling.datamodel.base_models import AcceleratorOptions, AcceleratorDevice

class DocumentParsingAgent:
    def __init__(self):
        pipeline_options = PipelineOptions()
        pipeline_options.ocr_options = RapidOcrOptions(backend="torch")
        pipeline_options.accelerator_options = AcceleratorOptions(device=AcceleratorDevice.CUDA)
        self.converter = DocumentConverter(pipeline_options=pipeline_options)
    
    def parse_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
        ext = file_path.lower().split('.')[-1]
        if ext == 'txt':
            return parse_sroie_box_file(file_path)
        elif ext in ['jpg', 'jpeg', 'png', 'pdf']:
            print(f"[Docling - GPU] Converting image: {file_path}")
            result = self.converter.convert(file_path)
            return result.document.export_to_markdown()
        else:
            raise NotImplementedError(f"Parsing for extension .{ext} not yet implemented.")
"""

with open("agents/parsing_agent.py", "w") as f:
    f.write(parsing_agent_code)
print("parsing_agent.py updated for GPU!")

### **Step 5: Set up Environment Variables**
Run this cell and securely paste your API keys when prompted. It will generate your `.env` file.

In [ ]:
import getpass

print("Enter your DEEPSEEK_API_KEY:")
deepseek_key = getpass.getpass()

print("Enter your SUPABASE_URL:")
supabase_url = getpass.getpass()

print("Enter your SUPABASE_KEY:")
supabase_key = getpass.getpass()

with open(".env", "w") as f:
    f.write(f"DEEPSEEK_API_KEY={deepseek_key}\n")
    f.write(f"SUPABASE_URL={supabase_url}\n")
    f.write(f"SUPABASE_KEY={supabase_key}\n")
print(".env file successfully created!")

### **Step 6: Run the Full Dataset at Lightning Speed! ⚡**
With the GPU enabled, the bottleneck is removed. Let's run `main.py`.

In [ ]:
!python main.py